# Funnel Analysis & Drop-Off Detection

This notebook demonstrates funnel analysis and drop-off detection using Pandas: defining sequential stages, computing drop-off and completion rates, visualizing user volume progression, ranking bottlenecks by monetary impact, and formulating evidence-based recommendations.

## Setup & Loading Data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure output directories exist
os.makedirs("../output", exist_ok=True)
os.makedirs("../data/raw", exist_ok=True)

RAW_DATA_PATH = "../data/raw/funnel_data.csv"

# Load funnel data
df = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded {len(df)} user sessions.")
df.head()

## Task 1: Define Funnel Stages and Count Users

We define a 6-stage sequential signup funnel and count the active users completing each step:
1. **Sign Up**: User initiates registration.
2. **Email Entered**: User submits an email address.
3. **Password Created**: User registers credentials.
4. **Email Verified**: User confirms email link.
5. **Payment Added**: User registers billing information.
6. **First Purchase**: User completes checkout.

Each step represents a strict prerequisite, guaranteeing that user count decreases or stays flat as they progress.

In [ ]:
stage1_signup = len(df[df['signup_completed'] == 1])
stage2_email = len(df[df['email_entered'] == 1])
stage3_password = len(df[df['password_created'] == 1])
stage4_verified = len(df[df['email_verified'] == 1])
stage5_payment = len(df[df['payment_added'] == 1])
stage6_purchase = len(df[df['first_purchase'] == 1])

stages = {
    'Sign Up': stage1_signup,
    'Email Entered': stage2_email,
    'Password Created': stage3_password,
    'Email Verified': stage4_verified,
    'Payment Added': stage5_payment,
    'First Purchase': stage6_purchase
}

print("=== FUNNEL STAGE COUNTS ===")
for k, v in stages.items():
    print(f"{k}: {v:,} users")

## Task 2: Compute Drop-Off Rate Between Stages

### Conversion Math:
- **Absolute Users Lost**: $U_{lost} = U_{before} - U_{after}$
- **Completion Rate**: $CR = \frac{U_{after}}{U_{before}} \times 100$
- **Drop-off Rate**: $DR = \frac{U_{lost}}{U_{before}} \times 100$ (representing percent of users who dropped *at this specific step*)

In [ ]:
stage_list = list(stages.values())
stage_names = list(stages.keys())

drop_off = []
for i in range(len(stage_list) - 1):
    users_before = stage_list[i]
    users_after = stage_list[i+1]
    users_lost = users_before - users_after
    drop_pct = (users_lost / users_before) * 100
    completion_pct = (users_after / users_before) * 100
    
    drop_off.append({
        'from_stage': stage_names[i],
        'to_stage': stage_names[i+1],
        'users_lost': users_lost,
        'completion_rate': completion_pct,
        'drop_rate': drop_pct
    })

funnel_df = pd.DataFrame(drop_off)
print("=== DETAILED DROP-OFF METRICS ===")
print(funnel_df)

# Find biggest drop by count
biggest_drop_count_idx = funnel_df['users_lost'].idxmax()
print(f"\nBiggest drop by user count: {funnel_df.loc[biggest_drop_count_idx, 'from_stage']} -> {funnel_df.loc[biggest_drop_count_idx, 'to_stage']} ({funnel_df.loc[biggest_drop_count_idx, 'users_lost']:,} lost)")

# Find highest drop rate percentage
highest_drop_rate_idx = funnel_df['drop_rate'].idxmax()
print(f"Highest drop rate percentage: {funnel_df.loc[highest_drop_rate_idx, 'from_stage']} -> {funnel_df.loc[highest_drop_rate_idx, 'to_stage']} ({funnel_df.loc[highest_drop_rate_idx, 'drop_rate']:.1f}% drop)")

## Task 3: Visualize Funnel

We plot a horizontal or vertical bar chart showing active volume at each stage, colored sequentially to denote the progression through checkout.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#3b82f6', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6', '#ec4899']
bars = ax.bar(stages.keys(), stages.values(), color=colors, edgecolor='black', alpha=0.85)

ax.set_ylabel('Active Users', fontsize=12, fontweight='bold')
ax.set_xlabel('Funnel Stages', fontsize=12, fontweight='bold')
ax.set_title('Signup Funnel Analysis: Volume & Progression', fontsize=14, fontweight='bold', pad=15)
ax.set_ylim(0, max(stages.values()) * 1.15)
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Annotate counts and percentages of the signup base
base_val = list(stages.values())[0]
for bar in bars:
    height = bar.get_height()
    pct_of_base = (height / base_val) * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height):,}\n({pct_of_base:.1f}%)",
        ha='center',
        va='bottom',
        fontweight='bold',
        fontsize=10
    )

plt.xticks(rotation=15, fontsize=10)
plt.tight_layout()
plt.savefig('../output/funnel_chart.png', dpi=300)
plt.show()

## Task 4: Calculate Business Impact of Each Drop-Off

To prioritize engineering resources, we calculate the monetary value lost at each step. We assign a baseline Customer Lifetime Value (LTV) of **$100** per customer. 
- **Formula**: `revenue_lost = users_lost * LTV`
- **Priority Threshold**: High if revenue lost > $100,000, else Medium.

In [ ]:
revenue_per_customer = 100.0
impact_analysis = []

for idx, row in funnel_df.iterrows():
    users_lost = row['users_lost']
    revenue_lost = users_lost * revenue_per_customer
    priority = 'HIGH' if revenue_lost > 100000 else 'MEDIUM'
    
    impact_analysis.append({
        'drop_point': f"{row['from_stage']} -> {row['to_stage']}",
        'users_lost': int(users_lost),
        'revenue_lost_val': revenue_lost,
        'revenue_impact': f"${revenue_lost:,.0f}",
        'priority': priority,
        'drop_rate_val': row['drop_rate']
    })

impact_df = pd.DataFrame(impact_analysis)
# Rank by business impact (revenue lost descending, then drop rate descending to break ties)
impact_df = impact_df.sort_values(['revenue_lost_val', 'drop_rate_val'], ascending=[False, False]).reset_index(drop=True)

print("=== RANKED BUSINESS IMPACT OF DROP-OFFS ===")
impact_df[['drop_point', 'users_lost', 'revenue_impact', 'priority']]

## Task 5: Actionable Recommendation

### Funnel Optimization Playbook

1. **Identify Bottleneck**: The transition **`Payment Added -> First Purchase`** is our most critical bottleneck. It accounts for a **50.0% drop rate**, losing **2,000 customers** representing **$200,000** in potential lifetime value.
2. **Hypotheses & Root Cause Analysis**:
   - *Checkout UX friction*: The final form may be overly complex or suffer from slow credit card processing speeds.
   - *Sticker Shock*: Unanticipated shipping fees, handling costs, or regional taxes are only exposed right before payment execution.
   - *Lack of Purchase Urgency*: Users add credit cards to reserve an account, but have no short-term hook or immediate incentive to checkout.
3. **Proposed Interventions**:
   - Introduce one-click payment integration (e.g. Apple Pay, Google Pay).
   - Trigger an automated push notification / email offer offering 10% off within 15 minutes of card entry if checkout is incomplete.
   - Guarantee free shipping or declare all taxes upfront during the email verification step.
4. **Expected Impact**:
   - Optimizing the checkout conversion rate by 10% (recovering 200 lost customers) will recover **$20,000** in monthly revenue.
5. **Success Criteria**:
   - Launch an A/B test of the simplified checkout screen.
   - Success is defined as a statistically significant increase of 5%+ in the checkout completion rate within 14 days.

## Cohort & Comparative Funnel Analysis

In operational analytics, funnels are rarely static. To expand this analysis, we can compare funnels across:
1. **Cohorts (Time-based)**: Compare signup drop-offs of users registering in January vs. February to see if software upgrades or seasonal campaigns improved checkout completion.
2. **Segments (User-based)**: Split the funnel by acquisition source (Organic Search vs. Paid Ads) or customer segments (Startup vs. SMB) to evaluate where marketing spend recovers the highest-converting traffic.